Save as: module3-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-03/module3-simulation-lab.ipynb)

# Lab 3 — Silicon Plant Managers: Technology Adoption vs. the NPV Benchmark
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Do LLM managers adopt a transparently positive-NPV
input the way the textbook says (always) or the way humans do (a minority
at baseline, with strong sensitivity to *decision timing*)?

**The task:** a fertilizer-style decision — pay \$10 per acre at planting,
receive +\$15 per acre at the next harvest (a 50% seasonal return;
break-even discount rate 50%/season, so the NPV rule says adopt in every
cell of this design).

**Human benchmark:** in the Western Kenya experiments (Duflo, Kremer &
Robinson 2011), baseline adoption is a minority of farmers, and a small
time-limited discount at harvest (cash in hand, decide now) raised
adoption by about as much as a 50% subsidy at planting (cash tight,
decide later). Timing, not price, was the binding margin.

**The tension to write about:** an agent population matching the NPV rule
is textbook-right and human-wrong; one matching humans needs a *reason*
not to adopt. Which did you get, and why? (PS3 Part C works the same
question on a stylized result.)

**Hypotheses to state before running (see handout):** H1 silicon adoption
exceeds human baselines; H2 cash-in-hand raises adoption (the DKR
direction); H3 the impatient persona adopts less, especially when cash is
tight (interaction).

**How to run:** no prior Python experience needed. Run the cells top to
bottom. Your required modification (handout) goes **only** in the cells
marked **CHANGE HERE**.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## Experimental design — CHANGE HERE (all four blocks live in this cell)

3 personas × 2 decision timings = 6 cells × 30 agents = 180 calls.

Personas are in **plain behavioral language** — no discount factors, no
Greek letters. If we wrote "your seasonal discount rate is 60%", the model
would compute the answer; that tests arithmetic, not behavior (the
robustness section asks you to try exactly that as a bait).

The timing treatment mirrors DKR: in `cash_in_hand`, the decision happens
right after harvest with money available and a time-limited offer; in
`cash_tight`, the same purchase is decided at planting time when money is
scarce. The prices and returns are identical — only the decision moment
and the agent's cash position differ.

Your required modification will usually mean **(1)** adding a factor to
`PERSONA_GRID` (e.g. `"financing": ["cash", "borrow_20pct"]` or an
information-source factor), **(2)** weaving it into `build_prompt`
*without changing any other wording across conditions*, and **(3)**
touching `parse_response` / `mock_response` only if the answer format
changes.


In [ ]:
PERSONAS = {                                     # CHANGE HERE (1 of 4)
    "patient_planner": (
        "You manage your farm like a business: you keep simple accounts, "
        "plan each season in advance, and when something is worth doing "
        "you arrange the money and do it on schedule."
    ),
    "impatient_procrastinator": (
        "You run your farm season to season. Money in hand tends to get "
        "spent on immediate needs, and paperwork or purchases that can "
        "wait usually do wait, even when you intend to get to them."
    ),
    "cautious_skeptic": (
        "You farm the way your family always has. New inputs and new "
        "advice have burned neighbors before, and you would rather have "
        "a sure modest harvest than gamble on promises."
    ),
}

PERSONA_GRID = {
    "persona": list(PERSONAS),
    "timing": ["cash_in_hand", "cash_tight"],   # the DKR manipulation
}

COST_PER_ACRE = 10
GAIN_PER_ACRE = 15


def build_prompt(persona: dict) -> str:          # CHANGE HERE (2 of 4)
    """Adoption decision prompt. Design rules: persona first, situation
    second, response format last; identical numbers in every condition;
    only the decision moment and cash position vary."""
    if persona["timing"] == "cash_in_hand":
        situation = (
            "You have just sold your harvest and have cash on hand. A "
            "supplier is at the market today with a one-day offer: order "
            f"fertilizer now for next season at ${COST_PER_ACRE} per acre, "
            "paid today, delivered at planting."
        )
    else:  # cash_tight
        situation = (
            "It is planting time. The harvest money from months ago has "
            "mostly gone to household needs, and cash is tight. The "
            f"supplier offers fertilizer at ${COST_PER_ACRE} per acre, "
            "paid now."
        )

    return (
        f"{PERSONAS[persona['persona']]}\n\n"
        f"{situation}\n\n"
        "On land like yours, this fertilizer typically raises harvest "
        f"revenue by about ${GAIN_PER_ACRE} per acre, received at the "
        "next harvest, one season after planting. You are deciding for "
        "one acre.\n\n"
        "Decide what you actually do (not what you intend to do "
        "eventually).\n\n"
        'Respond with ONLY a JSON object: {"adopt": true or false}.'
    )


def parse_response(text: str) -> "dict | None":  # CHANGE HERE (3 of 4)
    """Extract the decision. Return None on failure — never guess."""
    match = re.search(r'\{\s*"adopt"\s*:\s*(true|false)\s*\}',
                      text, re.IGNORECASE)
    if match:
        return {"adopt": match.group(1).lower() == "true"}
    return None


def mock_response(prompt: str) -> str:           # CHANGE HERE (4 of 4)
    """DRY_RUN only: synthetic placeholder answers so the notebook executes
    without API calls. NOT model behavior — these numbers are made up and
    deliberately mimic the human pattern (timing effects, persona
    interaction) so the analysis tables are legible."""
    cash_in_hand = "sold your harvest" in prompt
    if "like a business" in prompt:              # patient planner
        p = 0.90 if cash_in_hand else 0.80
    elif "usually do wait" in prompt:            # impatient procrastinator
        p = 0.70 if cash_in_hand else 0.25
    else:                                        # cautious skeptic
        p = 0.45 if cash_in_hand else 0.35
    return json.dumps({"adopt": random.random() < p})


## Run the experiment *(do not modify)*

Runs every persona × timing cell, logs parse failures instead of dropping
them, saves `lab3_results.csv` and `lab3_prompts_log.json` (required in
your write-up appendix). With `DRY_RUN = False` this makes ~180 API calls.


In [ ]:
def run_experiment() -> pd.DataFrame:
    keys = list(PERSONA_GRID)
    cells = [dict(zip(keys, combo)) for combo in itertools.product(*PERSONA_GRID.values())]
    print(f"{len(cells)} cells x {N_PER_CELL} agents = {len(cells) * N_PER_CELL} calls")

    rows = []
    for cell in cells:
        prompt = build_prompt(cell)
        for i in range(N_PER_CELL):
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=100,
                    temperature=TEMPERATURE,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = response.content[0].text
                decision = parse_response(raw)
            except Exception as err:            # log failures; never drop silently
                raw, decision = f"ERROR: {err}", None
                time.sleep(5)
            rows.append({
                **cell, "rep": i,
                "adopt": None if decision is None else decision["adopt"],
                "raw": raw, "model": MODEL, "temperature": TEMPERATURE,
            })
        done = [r for r in rows if r["adopt"] is not None]
        print(f"  cell {cell} done ({len(done)}/{len(rows)} parsed)")

    df = pd.DataFrame(rows)
    df.to_csv("lab3_results.csv", index=False)
    # Save exact prompts — required in the write-up appendix.
    with open("lab3_prompts_log.json", "w") as f:
        json.dump({str(c): build_prompt(c) for c in cells}, f, indent=2)
    return df


df = run_experiment()


## Analysis vs. the two benchmarks *(do not modify)*

Benchmark 1 — the NPV rule: with a 50% seasonal return, adoption should
be ~100% in every cell, for every persona, at any plausible discount rate.

Benchmark 2 — the human pattern (DKR 2011): minority baseline adoption,
with the harvest-time/cash-in-hand offer raising adoption substantially.

Matching Benchmark 1 means diverging from Benchmark 2, and vice versa —
the persona × timing table below is where you find out which happened.


In [ ]:
valid = df.dropna(subset=["adopt"]).copy()
valid["adopt"] = valid["adopt"].astype(bool)
print(f"Parse-failure rate: {df['adopt'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

print("\n=== Adoption rate by persona x timing ===")
table = (valid.groupby(["persona", "timing"])["adopt"]
         .agg(["mean", "count"]).round(3))
print(table)

print("\n=== Timing effect (cash_in_hand minus cash_tight), by persona ===")
rates = valid.groupby(["persona", "timing"])["adopt"].mean().unstack()
rates["timing_effect_pp"] = ((rates["cash_in_hand"] - rates["cash_tight"]) * 100).round(1)
print(rates.round(3))

print("\n=== Read against the benchmarks ===")
overall = valid["adopt"].mean()
print(f"Overall silicon adoption: {overall:.2f}")
print("NPV rule:                 ~1.00 in every cell (50% seasonal return)")
print("Human pattern (DKR):      minority baseline; timing moves adoption "
      "substantially")
print("\nIf your table matches the NPV rule, diagnose calculator capture. "
      "If it matches the human pattern, ask what gave the personas traction "
      "(and whether it survives the robustness checks).")


## Robustness — required in every lab

Rerun at least the `patient_planner × cash_in_hand` and
`impatient_procrastinator × cash_tight` cells with:

1. **a paraphrased prompt** (rewrite `build_prompt`'s wording, same
   content);
2. **a second model** (change `MODEL` in the setup cell);
3. **the explicit-parameterization bait** — add "your seasonal discount
   rate is 60%" to the persona and see whether behavior snaps to the
   exact NPV answer (at 60% > 50% break-even, the textbook says refuse).
   If it snaps, the model is solving, not behaving — what does that say
   about your main results?

**Limits of silicon subjects:** a transparently positive NPV may trigger
the model's calculator rather than its "behavior" — high adoption tells
you about arithmetic, not farmers. No LLM is actually short of cash at
planting; scarcity narratives bind only if the persona gives them
something to bind on. And a sim that fails to reproduce the adoption gap
still earns its keep debugging instrument wording and framing before the
human study spends real money (write-up Sections 5–6; PS3 Part C).
